<a href="https://colab.research.google.com/github/ShanshanHoo/agentic-research-paper-assistant/blob/main/Ingestion_IGN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import json
import requests
import xml.etree.ElementTree as ET
from urllib.parse import urljoin, urlparse

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

FEEDS_PAGE = "https://corp.ign.com/feeds"
OUTPUT_FILE = os.path.join(DATA_DIR, "ign_feeds.json")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; IGNFeedIngestor/1.0)"
}


def fetch_page(url: str) -> str:
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


def find_feed_links(html: str, base_url: str) -> list[str]:
    """
    Find likely RSS/XML/JSON feed links on the IGN feeds page.
    """
    candidates = set()

    # href="..."
    hrefs = re.findall(r'href=["\'](.*?)["\']', html, flags=re.IGNORECASE)
    for href in hrefs:
        full_url = urljoin(base_url, href)

        # likely feed patterns
        if any(
            token in full_url.lower()
            for token in ["/rss/", ".xml", "feed", ".json", "feeds"]
        ):
            candidates.add(full_url)

    # also catch raw URLs present in page text
    raw_urls = re.findall(r'https?://[^\s"\'<>]+', html, flags=re.IGNORECASE)
    for raw in raw_urls:
        if any(
            token in raw.lower()
            for token in ["/rss/", ".xml", "feed", ".json", "feeds"]
        ):
            candidates.add(raw)

    return sorted(candidates)


def strip_namespace(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def get_child_text(node, names: list[str]) -> str:
    """
    Return first matching child text by local tag name.
    """
    if node is None:
        return ""
    for child in node.iter():
        if strip_namespace(child.tag) in names and child.text:
            return child.text.strip()
    return ""


def parse_rss_or_atom(xml_text: str, source_url: str) -> list[dict]:
    """
    Parse RSS or Atom feeds into a normalized JSON structure.
    """
    root = ET.fromstring(xml_text)
    items = []

    root_name = strip_namespace(root.tag).lower()

    # RSS
    if root_name == "rss":
        channel = None
        for child in root:
            if strip_namespace(child.tag).lower() == "channel":
                channel = child
                break

        if channel is None:
            return items

        for item in channel:
            if strip_namespace(item.tag).lower() != "item":
                continue

            record = {
                "source_type": "ign_rss",
                "feed_url": source_url,
                "title": get_child_text(item, ["title"]),
                "link": get_child_text(item, ["link"]),
                "summary": get_child_text(item, ["description", "summary"]),
                "published": get_child_text(item, ["pubDate", "published", "updated"]),
                "author": get_child_text(item, ["author", "creator"]),
            }
            items.append(record)

    # Atom
    elif root_name == "feed":
        for entry in root:
            if strip_namespace(entry.tag).lower() != "entry":
                continue

            link = ""
            for child in entry:
                if strip_namespace(child.tag).lower() == "link":
                    link = child.attrib.get("href", link)

            record = {
                "source_type": "ign_atom",
                "feed_url": source_url,
                "title": get_child_text(entry, ["title"]),
                "link": link,
                "summary": get_child_text(entry, ["summary", "content"]),
                "published": get_child_text(entry, ["published", "updated"]),
                "author": get_child_text(entry, ["name", "author"]),
            }
            items.append(record)

    return items


def fetch_and_parse_feed(feed_url: str) -> list[dict]:
    """
    Download a feed URL and parse XML.
    If the endpoint is JSON, try loading it directly.
    """
    resp = requests.get(feed_url, headers=HEADERS, timeout=20)
    resp.raise_for_status()

    content_type = resp.headers.get("Content-Type", "").lower()
    text = resp.text.strip()

    # JSON feed
    if "json" in content_type or text.startswith("{") or text.startswith("["):
        data = resp.json()
        if isinstance(data, dict):
            return [data]
        if isinstance(data, list):
            return data
        return []

    # Otherwise treat as XML feed
    return parse_rss_or_atom(text, feed_url)


def dedupe_records(records: list[dict]) -> list[dict]:
    seen = set()
    out = []
    for r in records:
        key = (r.get("title", ""), r.get("link", ""))
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def fetch_ign_feeds(feeds_page: str = FEEDS_PAGE) -> list[dict]:
    html = fetch_page(feeds_page)
    feed_links = find_feed_links(html, feeds_page)

    print(f"Found {len(feed_links)} possible feed links.")

    all_records = []
    for url in feed_links:
        try:
            records = fetch_and_parse_feed(url)
            if records:
                print(f"Parsed {len(records)} items from: {url}")
                all_records.extend(records)
            else:
                print(f"No items parsed from: {url}")
        except Exception as e:
            print(f"Skipping {url} due to error: {e}")

    all_records = dedupe_records(all_records)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_records)} records to {OUTPUT_FILE}")
    return all_records


if __name__ == "__main__":
    fetch_ign_feeds()

Found 64 possible feed links.
Parsed 307 items from: http://feeds.feedburner.com/FireteamChatIgnsDestinyPodcast
Parsed 26 items from: http://feeds.feedburner.com/IGNHistoryAwesome
Parsed 20 items from: http://feeds.feedburner.com/IGNPS4Articles
Parsed 20 items from: http://feeds.feedburner.com/IGNPS4Reviews
Parsed 40 items from: http://feeds.feedburner.com/IGNPS4Videos
Parsed 20 items from: http://feeds.feedburner.com/IGNXboxOneArticles
Parsed 20 items from: http://feeds.feedburner.com/IGNXboxOneReviews
Parsed 40 items from: http://feeds.feedburner.com/IGNXboxOneVideos
Parsed 66 items from: http://feeds.feedburner.com/IgnUnfiltered
Parsed 7 items from: http://feeds.feedburner.com/ign-nintendo-switch-articles
No items parsed from: http://feeds.feedburner.com/ign-nintendo-switch-reviews
No items parsed from: http://feeds.feedburner.com/ign-nintendo-switch-videos
Parsed 20 items from: http://feeds.feedburner.com/ign/all
No items parsed from: http://feeds.feedburner.com/ign/comics-articles

In [2]:
import pandas as pd
import json

# Load the data from the output file
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Total number of records parsed: {len(df)}")
print(f"Number of unique feed URLs: {df['feed_url'].nunique()}")
print("\nDistribution of Source Types:")
print(df['source_type'].value_counts())

print("\nFirst 5 records:")
display(df.head())

Total number of records parsed: 5601
Number of unique feed URLs: 40

Distribution of Source Types:
source_type
ign_rss    5601
Name: count, dtype: int64

First 5 records:


,source_type,feed_url,title,link,summary,published,author
0,ign_rss,http://feeds.feedburner.com/FireteamChatIgnsDe...,Destiny 2: Bungie on Designing the Dread - Fir...,,This week Ben Wommack joins us to discuss Dest...,"Fri, 21 Jun 2024 21:00:00 -0000",IGN
1,ign_rss,http://feeds.feedburner.com/FireteamChatIgnsDe...,Destiny 2 The Final Shape is Everything We Hop...,,"Full Spoiler warning this episode! Destin, Tra...","Fri, 14 Jun 2024 20:36:54 -0000",IGN
2,ign_rss,http://feeds.feedburner.com/FireteamChatIgnsDe...,"Destiny 2: Right Now, We're Worried About Pris...",,The latest reveals about Destiny 2 Prismatic h...,"Sat, 11 May 2024 00:36:34 -0000",IGN
3,ign_rss,http://feeds.feedburner.com/FireteamChatIgnsDe...,Destiny 2 Needs to Fix This! Our Feedback For ...,,Destiny 2 feedback has been requested by Bungi...,"Fri, 03 May 2024 22:44:00 -0000",IGN
4,ign_rss,http://feeds.feedburner.com/FireteamChatIgnsDe...,Destiny 2 Sunsetting REMOVED & HUGE Power Chan...,,Destiny 2 sunsetting has been removed and with...,"Fri, 26 Apr 2024 19:59:59 -0000",IGN
